# Data Preparation & EDA

This project measures how generative AI is reshaping tech hiring between 2019 and 2026. Five data sources support that goal.

## Data Sources

1. **Hacker News "Who is hiring?" threads** (`data/raw/hn/`) — 93 monthly threads from January 2019 through September 2026, with 52,040 raw postings in total. Each posting is a single top-level comment written in free text. This is the main text corpus for the project.
2. **Multi-ATS postings, collected through Apify** (`data/raw/ats/`) — 4,409 postings pulled directly from company hiring systems (Greenhouse, Lever, Ashby) for a fixed list of AI-related companies. These postings arrive already structured, with company, role, salary range, and remote status as separate fields rather than free text.
3. **Indeed Hiring Lab trackers** (`data/raw/indeed/`) — public indexes of job-posting volume, covering roughly 1.8 million rows across five files: national, sector (47 industries), metro (593 areas), and state geographies for the US, plus a 9-country share-of-postings-mentioning-AI series. These provide an outside benchmark for whether patterns in the Hacker News and ATS data match the wider market.
4. **Bureau of Labor Statistics time series, pulled from the API** (`data/raw/bls/`) — 22 series across three surveys: JOLTS (job openings, hires, quits, layoffs), CES (payroll employment), and OEWS (employment and wages by occupation). These numbers serve as a macro-level check against trends seen in the posting data.
5. **BLS OEWS annual files, downloaded directly from the BLS website** (`data/raw/bls/oews/`) — one national workbook per year, 2019 through 2025. The OEWS API only ever exposes the current reference year, so these downloads are the only source of multi-year OEWS history.

Every cleaning step shown in this notebook calls code already written in `src/`; none of it is duplicated here.

## Data Cleaning

Each subsection below walks one source through its own pipeline: what shape the raw data arrives in, what changes at each step, and what gets flagged rather than silently dropped.

Every source below is walked through the same seven steps — loading and deduplication, text and field cleaning, field extraction, missing values, normalisation, outliers and invalid values, and discretisation and scaling — in that order, followed by a stage summary. Not every step applies to every source, and where one does not, that is stated explicitly rather than left as a silent gap. Across all five sources, nothing is dropped except exact duplicate rows; everything else that looks wrong, missing, or unusual is flagged in a new column so it can be filtered in or out per analysis, rather than being decided once here.

### Hacker News

Hacker News postings arrive as free text inside monthly hiring threads, with no structure beyond an informal "Company | Role | Location | ..." convention that most, but not all, posters follow. The pipeline below (`fetch_hn.py`, then `clean_hn.py`, then `clean_hn_stage2.py`) turns that into a structured table.

In [1]:
import json
import sys

sys.path.insert(0, "../src")

import pandas as pd

from clean_hn import RAW_DIR, SAMPLES_DIR, OUT_CSV as PARSED_CSV, REPORT_PATH as PARSE_REPORT, load_all, clean_text
from clean_hn_stage2 import OUT_CSV as CLEAN_CSV, REPORT_PATH as CLEAN_REPORT

pd.set_option("display.max_colwidth", 90)

#### Loading and deduplication

Hacker News marks some comments as deleted or dead, usually because a moderator removed them or the poster retracted the post. These comments carry no posting content — no company, no role, nothing usable. They are dropped before any other processing, so every later step works only with comments that could plausibly be a real job posting.

The cleaned table (`clean_df`) and its pipeline report (`report_lines`) are loaded once below and reused throughout the rest of this section. Companies repost the same hiring pitch across many of the 93 monthly threads, sometimes unchanged for years. This matters more for Hacker News than for the ATS data: an unflagged repost split across train and test sets would leak the same text to both. Each posting's first 400 characters are compared only against earlier postings from the *same* company — never across companies, and never all 48k postings against each other — using a string-similarity ratio, with anything above 90% treated as a repost of the earliest match. Nothing is dropped: `is_repost`, `repost_of`, and `repost_group` are added so the copies can be identified and excluded from a split later.

In [2]:
records = load_all()
print("\n".join(PARSE_REPORT.read_text().splitlines()[:3]))

loaded: 52040
after dead/deleted filter: 48296 (dropped 3744)
final rows written: 48296


In [3]:
dropped = [r for r in records if r.get("deleted") or r.get("dead")]
pd.DataFrame(dropped[:3])[["id", "by", "deleted", "dead", "text"]]

,id,by,deleted,dead,text
0,18808437,None,True,None,None
1,18807657,None,True,None,None
2,18807045,None,True,None,None


In [4]:
clean_df = pd.read_csv(CLEAN_CSV)
report_lines = CLEAN_REPORT.read_text().splitlines()

In [5]:
print(next(line for line in report_lines if line.startswith("is_repost")))
print(next(line for line in report_lines if line.startswith("repost groups")))

largest_group = clean_df["repost_group"].value_counts().idxmax()
clean_df.loc[clean_df["repost_group"] == largest_group, ["id", "month", "company_clean", "is_repost"]].head(6)

is_repost rows: 17538 (36.3%)
repost groups (2+ postings, excludes unmatched originals): 6567


,id,month,company_clean,is_repost
2779,19545345,2019-04,datadog,False
3095,19798986,2019-05,datadog,True
3837,20084359,2019-06,datadog,True
5011,20328971,2019-07,datadog,True
5734,20590752,2019-08,datadog,True
5982,20870931,2019-09,datadog,True


#### Text and field cleaning

Each posting arrives as HTML: entity codes such as `&#x27;` for an apostrophe, `<p>` tags marking paragraph breaks, and other markup mixed into the text. Left as-is, this markup makes the text hard to read and hard to search. This step decodes those entity codes, turns paragraph tags into line breaks first, strips any remaining tags, and collapses extra whitespace. Line breaks are preserved deliberately, because the first line of a posting usually carries the company and role — losing that structure would lose that information.

A separate text field is built for later modelling steps: the cleaned text, lowercased, with links and email addresses removed. Common words are left in place, since the modelling tools used later handle those automatically.

In [6]:
demo_raw = next(r["text"] for r in records if r.get("text") and "<p>" in r["text"])
print("BEFORE:\n", demo_raw[:300])
print("\nAFTER:\n", clean_text(demo_raw)[:300])

BEFORE:
 Y Combinator (yes, the people who run this site) | Full stack web | San Francisco | Onsite | Fulltime<p>Happy New Year, Hacker News!<p>Y Combinator has a small ~5 person team in San Francisco that builds all the software that runs YC.  We don&#x27;t hire for this team very often, but we&#x27;re look

AFTER:
 Y Combinator (yes, the people who run this site) | Full stack web | San Francisco | Onsite | Fulltime
Happy New Year, Hacker News!
Y Combinator has a small ~5 person team in San Francisco that builds all the software that runs YC. We don't hire for this team very often, but we're looking to hire a c


In [7]:
demo_row = clean_df.loc[clean_df["has_url"] & clean_df["has_email"]].iloc[0]
print("text_clean:\n", demo_row["text_clean"][:200])
print("\ntext_model:\n", demo_row["text_model"][:200])

text_clean:
 Y Combinator (yes, the people who run this site) | Full stack web | San Francisco | Onsite | Fulltime
Happy New Year, Hacker News!
Y Combinator has a small ~5 person team in San Francisco that builds 

text_model:
 y combinator (yes, the people who run this site) | full stack web | san francisco | onsite | fulltime happy new year, hacker news! y combinator has a small ~5 person team in san francisco that builds 


#### Field extraction

Most postings open with a line like `Company | Role | Location | ...`, with segments separated by vertical bars. The company name is almost always the first segment, and the role is almost always the second, so those two are read by position. Everything after that is read by what it looks like, not by where it sits, because the number of segments varies a great deal from posting to posting. The table below shows the same 20 sample postings before and after this step, with the raw text next to the fields pulled out of it.

In [8]:
raw_sample = pd.DataFrame(json.load(open(SAMPLES_DIR / "hn_raw_sample.json")))[["id", "text"]]
parsed_sample = pd.read_csv(SAMPLES_DIR / "hn_parsed_sample.csv")[["id", "company", "role", "location", "work_mode", "parse_ok"]]
before_after = raw_sample.merge(parsed_sample, on="id").head(8)
before_after["text"] = before_after["text"].str[:90]
before_after

,id,text,company,role,location,work_mode,parse_ok
0,18807019,"BuzzSumo | Infrastructure Engineer | REMOTE | Full-Time | <a href=""https:&#x2F;&#x2F;b...",BuzzSumo,Infrastructure Engineer,NaN,REMOTE,True
1,18807020,"GitLab | Engineering and Non-Engineering Roles | Remote Only | Full-time | <a href=""ht...",GitLab,Engineering and Non-Engineering Roles,NaN,Remote Only,True
2,18807021,"PlanGrid (YCW12) | San Francisco | Full-time, On-Site | Visa<p>We’re building software...",PlanGrid (YCW12),San Francisco,NaN,"Full-time, On-Site",True
3,18807031,"HealthPrize | Frontend Engineer | REMOTE or NYC, Norwalk, CT | Full-time | <a href=""ht...",HealthPrize,Frontend Engineer,"REMOTE or NYC, Norwalk, CT",NaN,True
4,18807034,"Fullstack.io | Book author | Remote | Part Time | <a href=""https:&#x2F;&#x2F;www.fulls...",Fullstack.io,Book author,NaN,Remote,True
5,18807035,BBC iPlayer | London | Fulltime | On Site | Back-End Software Engineer | 33k - 49k GBP...,BBC iPlayer,London,NaN,On Site,True
6,18807054,"Aquabyte | San Francisco, CA | Full Time | Software Engineer &#x2F; Head of Engineerin...",Aquabyte,"San Francisco, CA",NaN,NaN,True
7,18807059,"Gambit Research Ltd (<a href=""http:&#x2F;&#x2F;gambitresearch.com"" rel=""nofollow"">http...",Gambit Research Ltd (http://gambitresearch.com),"London, UK",NaN,ONSITE,True


#### Missing values

The header line does not always include the location or the work arrangement (remote, hybrid, onsite). When a header is missing one of these, the rest of the posting is searched for it instead. A value found this way is marked as coming from the body rather than the header, so it remains possible to tell how reliable each value is.

In [9]:
print("\n".join(CLEAN_REPORT.read_text().splitlines()[:2]))

recovered = clean_df.loc[clean_df["location_source"] == "body", ["id", "text_clean", "location", "location_source"]].head(5).copy()
recovered["text_clean"] = recovered["text_clean"].str[:110]
recovered

work_mode fill rate: 39.2% -> 91.2%
location fill rate: 53.9% -> 68.6%


,id,text_clean,location,location_source
4,18808169,"SpaceX | Hawthorne, CA; Redmond, WA; Vandenberg, CA; Cape Canaveral, FL; McGregor, TX ...","Hawthorne, CA",body
6,18808749,"TSM (Team SoloMid) & Blitz | Los Angeles | Onsite, relocation offered | Full-Time\nWe'...",Los Angeles,body
13,18807324,"Commure, Inc. | San Francisco, CA / Boston, MA | Rust Engineer | Fulltime | ONSITE\nWe...","San Francisco, CA",body
14,18808239,"Broad Institute of MIT and Harvard | Cambridge, MA | Software Engineer | INTERNS, ONSI...","Cambridge, MA",body
15,18807919,GAMEANALTICS | London (UK) | On-site and remote | https://gameanalytics.com/careers\nG...,London,body


#### Normalisation

The same employer posts nearly every month for years, often under slightly different spellings — "Datadog", "Datadog Inc.", "Datadog (YC W12)". Left alone, these would be counted as different companies. This step lowercases the name and removes punctuation, legal suffixes (Inc, LLC, and similar), and parenthetical notes, so the same employer collapses to one consistent value.

Common tech hubs appear under many names — "SF", "San Francisco", "SF Bay Area" — that all mean the same place. This step maps known variants to one standard "City, State" form and fills in the country where that can be determined with confidence. A location outside this known list is left exactly as written rather than guessed at.

In [10]:
print(next(line for line in report_lines if line.startswith("distinct companies")))

clean_df.loc[clean_df["company"] != clean_df["company_clean"], ["company", "company_clean"]].drop_duplicates().head(8)

distinct companies: 17943 -> 16401


,company,company_clean
0,"Y Combinator (yes, the people who run this site)",y combinator
1,United States Digital Service,united states digital service
2,Fullstack.io,fullstack io
3,Neos Insurance,neos insurance
4,SpaceX,spacex
5,Prima Assicurazioni (prima.it),prima assicurazioni
6,TSM (Team SoloMid) & Blitz,tsm blitz
7,PathAI,pathai


In [11]:
print(next(line for line in report_lines if line.startswith("distinct locations")))

clean_df["location_clean"].value_counts().head(12).rename_axis("location_clean").reset_index(name="count")

distinct locations: 9679 -> 1973


,location_clean,count
0,Remote,8112
1,"San Francisco, CA",6297
2,"New York, NY",3609
3,"London, UK",2173
4,"Berlin, Germany",1356
5,"Boston, MA",1139
6,"Seattle, WA",867
7,"Los Angeles, CA",737
8,"Austin, TX",653
9,"Toronto, Canada",622


#### Outliers and invalid values

Not every comment on a hiring thread is a real job posting — some are one-line questions, moderator notices, or complaints about a company. A posting is flagged as junk only when it is both short and failed to yield a company and role, since a short posting that still parsed cleanly likely has real structure. Postings that are empty, contain only a link, or match a moderator marker such as `[flagged]` are flagged regardless of length. Separately, the small number of unusually long postings — the top half of one percent by length — are flagged so they can be reviewed or set aside later. A percentile cutoff suits posting length here because there is no external reference point for how long a job posting should be — the only meaningful boundary is where the observed distribution itself gets unusually long.

In [12]:
idx = report_lines.index("10 is_junk examples:")
print("\n".join(report_lines[idx - 1 : idx + 11]))
print()
print(next(line for line in report_lines if line.startswith("text_len:")))
print(next(line for line in report_lines if line.startswith("is_outlier_len")))

is_junk rows: 233 (0.5%)
10 is_junk examples:
  'Anyone hiring interns'
  'Intern'
  'Kaam24 (Delhi, India) is hiring for several tech related Roles https://angel.co/'
  'IdeaFox.io, Full-Stack-Developer, Berlin, Germany, full time, onsite or remote'
  '[flagged]'
  '[flagged]'
  'This company is not very good at communicating.'
  "Time waste alert! You don't actually bother to respond to candidates."
  'Senior PHP Developer\nAgent Software Ltd\nManchester, UK\nFull-Time\nhttps://spectre'
  'Accenture is always hiring, got a degree or two? Hit me up'

text_len: min=0 median=976 p95=1973 p99.5=3251 max=7678
is_outlier_len rows (>p99.5): 242


#### Discretisation and scaling

The ATS data has `desc_len_band` and `salary_band`; Hacker News has no equivalent yet. `text_len_band` splits postings into short/medium/long terciles, with the cut points computed on non-junk postings only so the small number of junk rows do not skew the boundaries. `posting_frequency_band` looks at each company across the full seven years and buckets it by how many distinct months it posted in, from a one-off appearance to a persistent presence — a signal for how established a company's hiring is, independent of any single posting's text.

`text_len_scaled` standardises `text_len` to zero mean and unit variance, so posting length sits on the same scale as other numeric features used downstream. The mean and standard deviation are fitted on non-junk postings only, then applied to every row, so the transform is reproducible and no posting is dropped in the process.

In [13]:
print(next(line for line in report_lines if line.startswith("text_len_band boundaries")))
print(clean_df["text_len_band"].value_counts(dropna=False))
print()
print(next(line for line in report_lines if line.startswith("posting_frequency_band boundaries")))
print(clean_df["posting_frequency_band"].value_counts(dropna=False))

text_len_band boundaries (tercile, non-junk rows only): [38, 806, 1183, 7678]
text_len_band
short     16037
medium    16030
long      15996
NaN         233
Name: count, dtype: int64

posting_frequency_band boundaries: one_off=1, occasional=2-5, regular=6-20, persistent=21+ (distinct months posted per company_clean)
posting_frequency_band
regular       16328
occasional    13264
one_off        9935
persistent     8735
NaN              34
Name: count, dtype: int64


In [14]:
print(next(line for line in report_lines if line.startswith("text_len_scaled")))

before_after = clean_df[["id", "is_junk", "text_len", "text_len_scaled"]].head(5).copy()
before_after

text_len_scaled: z-score fitted on non-junk rows only, mean=1064.57, std=524.45


,id,is_junk,text_len,text_len_scaled
0,18809137,False,1888,1.570096
1,18807461,False,986,-0.149805
2,18807034,False,1806,1.413741
3,18807903,False,1293,0.435572
4,18808169,False,1036,-0.054467


#### Stage summary

The table below shows how many postings remain at each stage of the Hacker News pipeline, from the raw comments through to the fully cleaned dataset.

In [15]:
parsed_n = len(pd.read_csv(PARSED_CSV, usecols=["id"]))
summary = pd.DataFrame({
    "stage": ["raw", "after dead/deleted filter", "parsed", "cleaned"],
    "rows": [len(records), parsed_n, parsed_n, len(clean_df)],
})
summary

,stage,rows
0,raw,52040
1,after dead/deleted filter,48296
2,parsed,48296
3,cleaned,48296


### ATS Job Postings

These postings come directly from each company's applicant tracking system rather than free text, so the cleaning here is less about parsing structure out of prose and more about reconciling four overlapping exports, standardising values across three different ATS platforms, and handling a salary field that mixes hourly rates, annual salaries, and outright data-entry errors.

In [16]:
from clean_ats import (RAW_DIR as ATS_RAW_DIR, OUT_CSV as ATS_CSV, REPORT_PATH as ATS_REPORT,
                        COMPANY_MAP, SENIORITY_RULES, ROLE_RULES,
                        clean_description, classify_seniority, classify_role_family,
                        load_all as load_all_ats)
from clean_ats_stage2 import (OUT_CSV as ATS_FINAL_CSV, REPORT_PATH as ATS_FINAL_REPORT,
                               recover_city_region, normalize_location, location_primary)

ats_report = ATS_REPORT.read_text().splitlines()
ats_final_report = ATS_FINAL_REPORT.read_text().splitlines()
ats_raw_sample = pd.read_csv("../data/samples/ats_raw_sample.csv")
ats_clean_sample = pd.read_csv("../data/samples/ats_clean_sample.csv")
ats_final = pd.read_csv(ATS_FINAL_CSV)

#### Loading and deduplication

The raw export is four separate CSVs from Apify — two full company snapshots (Companies A and Companies B) and two smaller incremental batches (Batch B1 and Batch B2) run a few hours later to catch anything the first pass missed. That re-run means 115 job IDs show up in both an incremental batch and the snapshot it was topping up; the more recently scraped copy of each duplicate is kept, sorted by `scrapedAt`.

Some companies list the same role many times under different job IDs — Palantir's "Backend Software Engineer - Defense" appears three times, for instance, likely one opening per region or hiring cycle. A row is flagged `is_near_dup` when its company, normalised title, and first 300 characters of description all match another row. Nothing is dropped, since a genuine duplicate listing and a same-titled opening in a different location look identical under this rule, and only a human reviewer can tell them apart; 933 rows get the flag.

In [17]:
raw_ats, load_lines = load_all_ats()
print("\n".join(load_lines))
print(f"\nloaded: {len(raw_ats)} rows, unique jobId: {raw_ats['jobId'].nunique()}")

ats_clean_for_count = pd.read_csv(ATS_CSV, usecols=["jobId"])
print(f"after dedup: {len(ats_clean_for_count)} rows")

  Batch B1.csv: 317 rows
  Batch B2.csv: 163 rows
  Companies A dataset.csv: 3574 rows
  Companies B dataset.csv: 355 rows

loaded: 4409 rows, unique jobId: 4294
after dedup: 4294 rows


In [18]:
ats_final.loc[ats_final["is_near_dup"], ["company_clean", "title", "jobId"]].drop_duplicates(subset=["company_clean", "title"]).head(8)

,company_clean,title,jobId
1,Palantir,Backend Software Engineer - Application Development,10dfc8bc-99ad-4ca2-ab76-853cb90a92c2
3,Palantir,Backend Software Engineer - Defense,1345438c-ebfc-4fa5-b545-30c1414f317c
6,Palantir,Backend Software Engineer - Infrastructure,6fe5515f-f677-4d98-8ac2-1775a425f5e7
9,Palantir,Commercial Administrative Business Partner,2c45b359-0d00-4c68-b13d-b9f405efb739
12,Palantir,Commercial Contracts Specialist,45241c61-11af-45b5-86a0-a5302c028d6d
14,Palantir,Corporate Counsel,1a581b97-ce5b-4b4c-84b0-dd54cf8cfe14
17,Palantir,Deal Operations Administrator,157d4289-183f-4c4d-bc77-0dbacaba4612
21,Palantir,Deal Team - Business Affairs,16a1b500-13fe-4c22-ad89-372093b462da


#### Text and field cleaning

`descriptionText` arrives as plain prose (100% populated across all four files), unlike the raw HTML in `descriptionHtml`, but it still carries non-breaking spaces (`\xa0`), curly quotes, and em dashes left over from however each ATS platform rendered the posting. This step normalises those to their plain equivalents and collapses repeated whitespace.

In [19]:
demo = ats_raw_sample["descriptionText"].iloc[0]
print("BEFORE:\n", demo[:300])
print("\nAFTER:\n", clean_description(demo)[:300])

BEFORE:
 ABOUT US

At Sierra, we’re building a platform to enable every company in the world to build better, more human customer experiences with AI. We partner with industry leaders such as SoftBank, Uber, Rivian, CLEAR, and Sutter Health. We are primarily an in-person company based in San Francisco, with 

AFTER:
 ABOUT US
At Sierra, we're building a platform to enable every company in the world to build better, more human customer experiences with AI. We partner with industry leaders such as SoftBank, Uber, Rivian, CLEAR, and Sutter Health. We are primarily an in-person company based in San Francisco, with g


#### Field extraction

No usable seniority field exists in the raw data (`experienceLevel` is 0% populated across all four files), so seniority is derived from the title text using an ordered rule set. Later rules override earlier ones, so a title matching two rules resolves to the more senior one — "Senior Staff Engineer" matches `senior` first, then `staff`, and `staff` wins.

A second ordered rule set buckets each title into a role family by regex over the title text. Order matters here too, but in the opposite direction from seniority: the first match wins, so `forward_deployed` is checked before `solutions` and `software_eng`, or "Forward Deployed Software Engineer" would fall into `software_eng` instead.

In [20]:
display(pd.DataFrame({"seniority": [r[0] for r in SENIORITY_RULES], "pattern": [r[1].pattern for r in SENIORITY_RULES]}))
print("classify_seniority('Senior Staff Engineer') ->", classify_seniority("Senior Staff Engineer"))

idx = ats_report.index("seniority distribution:")
print("\n" + "\n".join(ats_report[idx : idx + 8]))

,seniority,pattern
0,intern,(?i:intern(ship)?)
1,new_grad,(?i:new\s?grad(uate)?|university\s?grad|early career)
2,junior,(?i:junior|jr\.?|associate)|\bI\b
3,senior,(?i:senior|sr\.?)|\bII\b|\bIII\b
4,staff,(?i:staff|principal|distinguished|architect)
5,lead,(?i:lead|manager|head of|director|vp|chief)


classify_seniority('Senior Staff Engineer') -> staff

seniority distribution:
  mid: 1703
  lead: 1435
  staff: 626
  senior: 401
  intern: 62
  new_grad: 38
  junior: 29


In [21]:
display(pd.DataFrame({"role_family": [r[0] for r in ROLE_RULES], "pattern": [r[1].pattern for r in ROLE_RULES]}))

idx = ats_report.index("role_family distribution:")
print("\n" + "\n".join(ats_report[idx : idx + 11]))

,role_family,pattern
0,forward_deployed,forward.?deploy|\bFDE\b
1,solutions,solutions? (eng|arch|consult)|delivery|deployment strateg|technical account
2,applied_ai,applied (ai|ml|scien)|ai engineer|ml engineer|machine learning eng
3,research,research (scien|eng)
4,software_eng,software engineer|backend|frontend|infrastructure|platform eng|\bsre\b
5,data,data (scien|eng|analy)
6,product,product manager|product design
7,sales_gtm,account exec|sales|\bgtm\b|partnership|business development
8,recruiting,recruit|talent|people ops



role_family distribution:
  other: 1781
  software_eng: 839
  sales_gtm: 603
  solutions: 327
  forward_deployed: 245
  product: 177
  applied_ai: 113
  research: 82
  data: 68
  recruiting: 59


#### Missing values

Most of the highly-missing fields are not gaps waiting to be filled: `salaryMin`/`salaryMax` are missing because Greenhouse-listed companies simply do not expose a compensation field at all, and `employmentType` is 100% populated on Ashby and Lever but 0% on Greenhouse. These are left null and flagged (`has_salary`, `has_department`) rather than imputed, since the missingness pattern is structural — platform-driven, not random.

In [22]:
missing = (ats_final.isna().mean() * 100).round(1).sort_values(ascending=False)
missing[missing > 0].head(10).rename("pct_missing").rename_axis("column").reset_index()

,column,pct_missing
0,salaryInterval,95.2
1,salary_band,64.6
2,salary_annual_min,64.6
3,salary_suspect,64.6
4,salary_is_usd,64.6
5,salary_currency,64.6
6,salary_annual_max,64.6
7,salaryMin,64.6
8,salaryMax,64.6
9,salaryCurrency,64.6


#### Normalisation

The same 19 companies appear under inconsistent casing depending on which ATS platform scraped them — `openai`, `palantir`, `Scale AI`, `Anthropic`. A hand-written mapping (`COMPANY_MAP`) is more reliable here than a title-casing rule, which would mangle names like "OpenAI" and "Scale AI".

1,521 of 4,294 rows carry a `salaryMin`, but `salaryInterval` — whether that figure is hourly or annual — is blank on 1,314 of them. Values range from a few dollars up into the hundreds of thousands, so the column mixes annual salaries with hourly rates, and the interval has to be inferred from magnitude: under \$1,000 reads as hourly, \$1,000 and up reads as annual. Hourly figures are then multiplied by 2,080 (a working year) to get an annual equivalent.

`city` and `region` are populated on only 35% and 33% of rows directly, but the free-text `location` field (99.98% populated) usually names a city even when the structured fields do not. `recover_city_region` pulls a city/region pair out of it where possible, filtering out region-only descriptors like "Remote" or "West Coast - United States" rather than mis-attributing them as a city — that recovery lifts the city fill rate from 35.0% to 91.6%. Separately, the free-text `location` string is mapped to one of 21 canonical hub names where it matches a known hub, collapsing 430 raw location strings down to 151 canonical values.

In [23]:
print(f"{len(COMPANY_MAP)} companies mapped\n")
ats_clean_sample[["companyName", "company_clean"]].drop_duplicates().head(8)

19 companies mapped



,companyName,company_clean
0,palantir,Palantir
1,openai,OpenAI
10,sierra,Sierra
14,cohere,Cohere
16,harvey,Harvey
19,perplexity,Perplexity


In [24]:
idx = ats_report.index("salary: 1521 / 4294 rows have salaryMin")
print("\n".join(ats_report[idx : idx + 5]))

salary: 1521 / 4294 rows have salaryMin
  salaryInterval given blank/year/hour: 1314 / 205 / 2
  interval inferred as hourly: 5, annual: 1516
  currency: USD=1469, GBP=43, EUR=9
  salary_is_usd True/False: 1469 / 52


In [25]:
for raw in ["Abilene, TX", "Remote-Friendly (Travel-Required)", "Hybrid - San Francisco, New York City", "US - Remote"]:
    primary = location_primary(raw)
    print(f"{raw!r:45} -> city/region: {recover_city_region(primary)}, canonical: {normalize_location(primary)!r}")

print()
idx = next(i for i, l in enumerate(ats_final_report) if l.startswith("city fill rate:"))
print("\n".join(ats_final_report[idx : idx + 2]))


'Abilene, TX'                                 -> city/region: ('Abilene', 'TX'), canonical: 'Abilene, TX'
'Remote-Friendly (Travel-Required)'           -> city/region: (None, None), canonical: 'Remote'
'Hybrid - San Francisco, New York City'       -> city/region: ('San Francisco', 'New York City'), canonical: 'San Francisco, CA'
'US - Remote'                                 -> city/region: (None, None), canonical: 'Remote'

city fill rate: 35.0% -> 91.6% (recovered from location)
region fill rate: 32.7% -> 78.5% (recovered from location)


In [26]:
ats_final["location_clean"].value_counts().head(8).rename_axis("location_clean").reset_index(name="count")

,location_clean,count
0,"San Francisco, CA",1754
1,"New York, NY",444
2,"London, UK",310
3,Remote,249
4,"Washington, DC",195
5,Singapore,106
6,"Bengaluru, India",98
7,"Tokyo, Japan",87


#### Outliers and invalid values

Values that are still implausible after standardising to an annual figure — under \$20,000 or above \$1,000,000 — are flagged `salary_suspect` rather than dropped; 12 rows get that flag, including a "Talent Coordinator" row explicitly tagged annual with a raw value of \$32. Domain rules suit salary better than a statistical cutoff here, since what counts as a plausible salary is set by real-world compensation norms, not by the shape of this particular dataset's distribution.

In [27]:
idx = ats_report.index("salary: 1521 / 4294 rows have salaryMin")
print(ats_report[idx + 5])
print()
ats_final.loc[ats_final["salary_suspect"] == True,
              ["title", "salaryMin", "salary_interval_inferred", "salary_annual_min"]].head(10)

  salary_suspect True/False: 12 / 1509



,title,salaryMin,salary_interval_inferred,salary_annual_min
29,Deployment Strategist,2400.0,year,2400.0
92,Forward Deployed Software Engineer,2800.0,year,2800.0
275,Talent Coordinator,32.0,year,32.0
276,Talent Coordinator,32.0,year,32.0
305,"Year at Palantir - Forward Deployed Software Engineer, Internship - Commercial",5900.0,year,5900.0
306,"Year at Palantir - Forward Deployed Software Engineer, Internship - Commercial",5900.0,year,5900.0
307,"Year at Palantir - Forward Deployed Software Engineer, Internship - USG",5900.0,year,5900.0
308,"Year at Palantir - Forward Deployed Software Engineer, Internship - USG",5900.0,year,5900.0
309,"Year at Palantir - Software Engineer, Internship",5900.0,year,5900.0
1004,"Research Engineer, Privacy",0.0,hour,0.0


#### Discretisation and scaling

Two banded fields are added for later grouping: `desc_len_band` splits postings into short/medium/long terciles by description length, and `salary_band` splits the rows that have a salary into quartiles of `salary_annual_min`. Both are computed with `pandas.qcut`, so each band holds roughly the same number of rows.

In [28]:
idx = next(i for i, l in enumerate(ats_final_report) if l.startswith("desc_len_band boundaries:"))
print("\n".join(ats_final_report[idx : idx + 11]))


desc_len_band boundaries: [1018, 5619, 7116, 16912]
  desc_len_band
  short     1432
  medium    1431
  long      1431
salary_band boundaries: [0, 150000, 200000, 257000, 510000]
  salary_band
  Q1    416
  Q3    389
  Q4    369
  Q2    347


#### Stage summary

Row counts for every stage of the ATS pipeline — loaded, deduplicated, and cleaned — are shown in the consolidated Pipeline Summary below, alongside the other four sources.

### Indeed Hiring Lab

Indeed's Hiring Lab publishes job-posting indexes as several separate, already-tidy CSVs rather than one combined table. The cleaning here is reshaping and validation — parsing dates, classifying sectors, and checking for gaps and range violations — rather than repair, since the source data is already clean.

In [29]:
from clean_indeed import REPORT_PATH as INDEED_REPORT, SECTOR_GROUP, classify_sector

indeed_report = INDEED_REPORT.read_text().splitlines()
indeed_ai_share = pd.read_csv("../data/processed/indeed_ai_share.csv")
indeed_national = pd.read_csv("../data/processed/indeed_national.csv")
indeed_sector = pd.read_csv("../data/processed/indeed_sector.csv")
indeed_metro = pd.read_csv("../data/processed/indeed_metro.csv")
indeed_state = pd.read_csv("../data/processed/indeed_state.csv")

#### Loading and deduplication

| file | what it covers |
|---|---|
| `indeed_ai_share.csv` | share of postings mentioning AI, 9 countries, daily |
| `indeed_national.csv` | overall US posting-volume index, seasonally adjusted and not — not one of the four files named in the original brief, but it comes along for free from the same `US/*.csv` glob that picks up the other three, so it is kept and flagged rather than silently dropped |
| `indeed_sector.csv` | posting-volume index for 47 sectors, total vs. new postings |
| `indeed_metro.csv` | posting-volume index for 593 US metro areas |
| `indeed_state.csv` | posting-volume index for 50 states + DC |

Every file is also checked for duplicate rows: across all five files, no exact or key-level duplicates exist anywhere.

In [30]:
for name, df in [("ai_share", indeed_ai_share), ("national", indeed_national), ("sector", indeed_sector),
                 ("metro", indeed_metro), ("state", indeed_state)]:
    print(f"{name}: {len(df):,} rows")

ai_share: 24,921 rows
national: 4,802 rows
sector: 225,694 rows
metro: 1,423,793 rows
state: 122,451 rows


In [31]:
lines = [l.strip() for l in indeed_report if l.strip().startswith("duplicates:")]
print("duplicates:")
print("  " + "\n  ".join(lines))

duplicates:
  duplicates: 0 exact duplicate rows dropped; 0 keys have conflicting values across rows (NOT dropped, needs a decision)
  duplicates: 0 exact duplicate rows dropped; 0 keys have conflicting values across rows (NOT dropped, needs a decision)
  duplicates: 0 exact duplicate rows dropped; 0 keys have conflicting values across rows (NOT dropped, needs a decision)
  duplicates: 0 exact duplicate rows dropped; 0 keys have conflicting values across rows (NOT dropped, needs a decision)
  duplicates: 0 exact duplicate rows dropped; 0 keys have conflicting values across rows (NOT dropped, needs a decision)


#### Text and field cleaning

Indeed's five files arrive as already-tidy, structured numeric series — dates, category labels, and index values — with no free text to clean and no fields to extract, so this step and Field extraction are skipped.

#### Missing values

Every file's date/series panel is checked for missing steps by comparing actual row counts against `n_dates x n_series`, rather than trusting that the source export is complete. Across all five files, every series runs gap-free from its first date to its last.

In [32]:
lines = [l.strip() for l in indeed_report if l.strip().startswith("date gaps:")]
print("date gaps:")
print("  " + "\n  ".join(lines))

date gaps:
  date gaps: 0 missing D steps across 0 of 9 series (a missing month/day here is distinct from a null value in a present row)
  date gaps: 0 missing D steps across 0 of 2 series (a missing month/day here is distinct from a null value in a present row)
  date gaps: 0 missing D steps across 0 of 94 series (a missing month/day here is distinct from a null value in a present row)
  date gaps: 0 missing D steps across 0 of 593 series (a missing month/day here is distinct from a null value in a present row)
  date gaps: 0 missing D steps across 0 of 51 series (a missing month/day here is distinct from a null value in a present row)


#### Normalisation

The sector file names 47 detailed sectors. The original brief specified 44 and named "Information Design & Documentation" as one of four AI-exposed sectors; that sector does not exist in the current data, so it is documented as missing rather than silently ignored. Three sectors are marked `ai_exposed` — Software Development, Data & Analytics, IT Systems & Solutions — and everything else defaults to `control`. The mapping lives in a plain dict specifically so a sector is easy to move from one group to the other.

Every Indeed index column — `indeed_job_postings_index` in the sector/metro/state files, `indeed_job_postings_index_sa`/`_nsa` in the national file — is set to 100 on February 1, 2020 for every series independently. That makes levels comparable across sectors, metros, or time *within* a series, but the numbers are not counts of postings and are not comparable in absolute terms between, say, a niche sector and a huge one.

In [33]:
sectors_found = sorted(set(indeed_sector["display_name"]))
print(f"{len(sectors_found)} sectors found (brief specifies 44)")
print("named ai_exposed in the brief but absent from this data:", [s for s in SECTOR_GROUP if s not in sectors_found])
print()
pd.DataFrame({"sector": sectors_found, "group": [classify_sector(s) for s in sectors_found]}).sort_values(["group", "sector"]).reset_index(drop=True)

47 sectors found (brief specifies 44)
named ai_exposed in the brief but absent from this data: ['Information Design & Documentation']



,sector,group
0,Data & Analytics,ai_exposed
1,IT Systems & Solutions,ai_exposed
2,Software Development,ai_exposed
3,Accounting,control
4,Administrative Assistance,control
5,Architecture,control
6,Arts & Entertainment,control
7,Aviation,control
8,Banking & Finance,control
9,Childcare,control


In [34]:
indeed_sector.loc[(indeed_sector["date"] == "2020-02-01") & (indeed_sector["variable"] == "total postings"),
                   ["display_name", "indeed_job_postings_index"]].head(6)

,display_name,indeed_job_postings_index
0,Accounting,100.0
2,Retail,100.0
4,Electrical Engineering,100.0
6,Sales,100.0
8,Hospitality & Tourism,100.0
9,Nursing,100.0


#### Outliers and invalid values

Anything outside [0, 500] on an index column is flagged `..._suspect` as implausible for a Feb-2020-based index. Six rows trip that flag in the sector file and six in the metro file — both turn out to be real, verifiable spikes rather than data errors: Aviation `new postings` hit 531 in early October 2023, and the Findlay, OH metro hit 538 in July 2025. A fixed plausible range suits an index like this because every series is rebased to the same 100 at the same date, so a value many multiples above that shared baseline is implausible regardless of which sector or metro it comes from.

In [35]:
indeed_sector.loc[indeed_sector["indeed_job_postings_index_suspect"],
                   ["date", "display_name", "variable", "indeed_job_postings_index"]]

,date,display_name,variable,indeed_job_postings_index
125850,2023-10-01,Aviation,new postings,510.23
125937,2023-10-02,Aviation,new postings,519.75
125976,2023-10-03,Aviation,new postings,531.33
126117,2023-10-04,Aviation,new postings,524.83
126194,2023-10-05,Aviation,new postings,519.53
126281,2023-10-06,Aviation,new postings,503.88


In [36]:
indeed_metro.loc[indeed_metro["indeed_job_postings_index_suspect"],
                  ["date", "metro_name", "metro_state", "indeed_job_postings_index"]]

,date,metro_name,metro_state,indeed_job_postings_index
1175914,2025-07-06,Findlay,OH,504.59
1176461,2025-07-07,Findlay,OH,512.41
1176978,2025-07-08,Findlay,OH,519.42
1177399,2025-07-09,Findlay,OH,526.09
1177720,2025-07-10,Findlay,OH,532.12
1178698,2025-07-11,Findlay,OH,538.11


#### Discretisation and scaling

Indeed's series are already index values on a common, Feb-2020-based scale; no additional binning or standardisation is applied on top of that.

#### Stage summary

Row counts for all five Indeed files are shown in the consolidated Pipeline Summary below, alongside the other four sources.

### BLS Time Series

The Bureau of Labor Statistics API returns three separate surveys — JOLTS, CES, and OEWS — each as deeply nested JSON rather than a flat table. All three go through the same flattening and cleaning pipeline, since they share the same `Results -> series -> data` shape.

In [37]:
from clean_bls import (RAW_DIR as BLS_RAW_DIR, REPORT_PATH as BLS_REPORT,
                        period_to_date, flatten, latest_file)

bls_report = BLS_REPORT.read_text().splitlines()
bls_jolts = pd.read_csv("../data/processed/bls_jolts.csv", parse_dates=["date"])
bls_ces = pd.read_csv("../data/processed/bls_ces.csv", parse_dates=["date"])
bls_oews = pd.read_csv("../data/processed/bls_oews.csv", parse_dates=["date"])

#### Loading and deduplication

Each API response nests as `Results -> series -> [{seriesID, data: [{year, period, value, footnotes}]}]`. `flatten()` walks that structure into one row per series/month. Series names come from a small local catalog file (`_series_catalog.json`) rather than the API's own inline catalog data — JOLTS returns "Unable to get Catalog Data" for every one of its 12 series, which is a gap on BLS's side, not a problem with the numbers themselves. No exact duplicate or key-conflicting (series, date) rows appear in any of the three surveys once flattened.

In [38]:
jolts_path = latest_file("jolts_*.json")
flat = flatten(jolts_path)
flat.head(6)

,series_id,series_name,year,period,value_raw,footnotes
0,JTS000000000000000JOL,Total nonfarm: job openings,2026,M07,7271,P: preliminary
1,JTS000000000000000JOL,Total nonfarm: job openings,2026,M06,7182,
2,JTS000000000000000JOL,Total nonfarm: job openings,2026,M05,7537,
3,JTS000000000000000JOL,Total nonfarm: job openings,2026,M04,7585,
4,JTS000000000000000JOL,Total nonfarm: job openings,2026,M03,6887,
5,JTS000000000000000JOL,Total nonfarm: job openings,2026,M02,6922,


#### Text and field cleaning

BLS's API pull arrives as structured numeric series — series ID, period, and value — with no free text to clean and no fields to extract from it, so this step and Field extraction are skipped.

#### Missing values

BLS's API pull has no missing values and no gaps in any of the 22 series across all three surveys — every month between each series' first and last date is present.

#### Normalisation

JOLTS and CES use `M01`-`M12` for calendar months and `M13` for an annual-average row mixed into the same series; that annual-average row is excluded from the monthly output (flagged, with the count reported) since a single yearly average does not belong in a monthly panel. OEWS instead marks its one annual figure per series as `A01` — treated as a valid date (January 1 of the reference year) rather than being dropped, since dropping it would leave the OEWS output empty.

In [39]:
for year, period in [("2026", "M07"), ("2020", "M13"), ("2025", "A01")]:
    print(f"{year} {period} -> {period_to_date(year, period)}")
print()
print("M13 rows flagged and excluded, per survey:")
print("\n".join(l for l in bls_report if l.startswith("M13")))

2026 M07 -> 2026-07-01 00:00:00
2020 M13 -> None
2025 A01 -> 2025-01-01 00:00:00

M13 rows flagged and excluded, per survey:
M13 (annual average) rows flagged and excluded: 0
M13 (annual average) rows flagged and excluded: 0
M13 (annual average) rows flagged and excluded: 0


#### Outliers and invalid values

Each series gets its own z-score — not pooled across series, since JOLTS levels for "Total nonfarm" and for "Information" sit on completely different scales — and anything beyond 3 standard deviations is flagged `is_outlier`. The flagged JOLTS rows cluster in March-April 2020, the initial COVID layoff wave, which is a reassuring sign that the flag is catching real events rather than noise. A fixed per-series threshold suits time series data like this because each series has its own stable scale and variance, so a single pooled cutoff would flag high-volume series constantly and low-volume series never.

In [40]:
bls_jolts.loc[bls_jolts["is_outlier"], ["series_id", "series_name", "date", "value"]].sort_values("date").head(8)

,series_id,series_name,date,value
340,JTS000000000000000LDL,Total nonfarm: layoffs and discharges,2020-03-01,12985
896,JTS510000000000000LDL,Information: layoffs and discharges,2020-03-01,191
1452,JTS540099000000000LDL,Professional and business services: layoffs and discharges,2020-03-01,1260
63,JTS000000000000000HIL,Total nonfarm: hires,2020-04-01,4029
341,JTS000000000000000LDL,Total nonfarm: layoffs and discharges,2020-04-01,9170
897,JTS510000000000000LDL,Information: layoffs and discharges,2020-04-01,169
1453,JTS540099000000000LDL,Professional and business services: layoffs and discharges,2020-04-01,1228
64,JTS000000000000000HIL,Total nonfarm: hires,2020-05-01,8133


#### Discretisation and scaling

BLS's time series values are levels and rates already on their natural units; no additional binning or scaling is applied at this stage.

#### Stage summary

The OEWS API exposes only the current reference year per series — one row per series, all dated 2025 — rather than any history. That is expected behaviour for this survey, not a data problem, and it is exactly why a separate multi-year OEWS pipeline (the next section) exists at all.

In [41]:
bls_oews[["series_id", "series_name", "date", "value"]]

,series_id,series_name,date,value
0,OEUN000000000000015122101,Computer and Information Research Scientists: employment,2025-01-01,37200
1,OEUN000000000000015122113,Computer and Information Research Scientists: median annual wage,2025-01-01,140300
2,OEUN000000000000015125201,Software Developers: employment,2025-01-01,1687890
3,OEUN000000000000015125213,Software Developers: median annual wage,2025-01-01,135980
4,OEUN000000000000015129901,"Computer Occupations, All Other: employment",2025-01-01,435370
5,OEUN000000000000015129913,"Computer Occupations, All Other: median annual wage",2025-01-01,116580
6,OEUN000000000000015205101,Data Scientists: employment,2025-01-01,262440
7,OEUN000000000000015205113,Data Scientists: median annual wage,2025-01-01,120230


### OEWS Annual Files

The OEWS API pull above only ever carries the current reference year, so it cannot show whether Software Developer wages or Data Scientist employment have been trending up or down since 2019. The BLS website separately publishes one full national workbook per year, and stacking seven of those (2019-2025) is the only way to build that trend.

#### Loading and deduplication

The API pull has exactly one row per OEWS series, always dated to the current reference year. The annual workbooks are the only way to see, for example, Software Developer employment or median wage move across multiple years. Once the seven yearly workbooks are stacked, no exact duplicate rows or conflicting (occupation code, year) keys appear.

In [42]:
from clean_oews_files import RAW_DIR as OEWS_XLSX_DIR, REPORT_PATH as OEWS_TREND_REPORT, load_year

oews_trend_report = OEWS_TREND_REPORT.read_text().splitlines()
oews_trend = pd.read_csv("../data/processed/oews_trend.csv")

In [43]:
print("API pull (bls_oews.csv) -- one row per series, current year only:")
display(bls_oews.loc[bls_oews["series_name"].str.contains("Software Developers"), ["series_name", "date", "value"]])

print("\nAnnual files (oews_trend.csv) -- full history:")
oews_trend.loc[oews_trend["occ_code"] == "15-1252", ["occ_code", "occ_title", "year", "tot_emp", "a_median"]]

API pull (bls_oews.csv) -- one row per series, current year only:


,series_name,date,value
2,Software Developers: employment,2025-01-01,1687890
3,Software Developers: median annual wage,2025-01-01,135980



Annual files (oews_trend.csv) -- full history:


,occ_code,occ_title,year,tot_emp,a_median
16,15-1252,Software Developers,2021,1364180,120730
17,15-1252,Software Developers,2022,1534790,127260
18,15-1252,Software Developers,2023,1656880,132270
19,15-1252,Software Developers,2024,1654440,133080
20,15-1252,Software Developers,2025,1687890,135980


#### Text and field cleaning

OEWS's annual workbooks arrive as structured columns — occupation code, title, employment, and wage figures — with no free text to clean; the one piece of per-value parsing needed, decoding the `*`/`#` suppression markers, is covered under Missing values below, since it is really about recovering which values are genuinely known versus withheld.

#### Missing values

Six target SOC codes are meant to track AI-adjacent computer occupations over time, but the 2018 SOC revision — which BLS phased into OEWS starting with the 2021 reference year — means not all six exist in every file. `Software Developers` (15-1252) and `Data Scientists` (15-2051) do not exist before 2021, since those occupations were not broken out under the old classification. `Database and Network Administrators` (15-1245) is the mirror case: it exists only in 2019-2020 and was split into two separate codes from 2021 onward, so no single code covers the full 2019-2025 span for that occupation. Both gaps are reported rather than silently interpolated or backfilled.

BLS uses `*` to mean a value was withheld for a small or unreliable sample, and `#` to mean a wage estimate above the topcode of \$239,200. One row in this data hits the topcode: Computer and Information Research Scientists in 2021, where the 90th-percentile annual wage is `#`. That figure is converted to null rather than silently reading as ordinary missing data, and the row is kept and flagged `wage_topcoded=True` rather than dropped.

In [44]:
idx = oews_trend_report.index("=== target occupation coverage (2018 SOC revision breaks some codes) ===")
print("\n".join(oews_trend_report[idx : idx + 9]))

=== target occupation coverage (2018 SOC revision breaks some codes) ===
  2019: found ['15-1211', '15-1221', '15-1245', '15-1299'], MISSING ['15-1252', '15-2051']
  2020: found ['15-1211', '15-1221', '15-1245', '15-1299'], MISSING ['15-1252', '15-2051']
  2021: found ['15-1211', '15-1221', '15-1252', '15-1299', '15-2051'], MISSING ['15-1245']
  2022: found ['15-1211', '15-1221', '15-1252', '15-1299', '15-2051'], MISSING ['15-1245']
  2023: found ['15-1211', '15-1221', '15-1252', '15-1299', '15-2051'], MISSING ['15-1245']
  2024: found ['15-1211', '15-1221', '15-1252', '15-1299', '15-2051'], MISSING ['15-1245']
  2025: found ['15-1211', '15-1221', '15-1252', '15-1299', '15-2051'], MISSING ['15-1245']
  NOTE: 15-1245 BLS title ['Database Administrators and Architects'] != brief's label 'Database and Network Administrators' -- BLS title kept


In [45]:
oews_trend.loc[oews_trend["wage_topcoded"], ["occ_code", "occ_title", "year", "a_median", "a_mean", "a_pct90"]]

,occ_code,occ_title,year,a_median,a_mean,a_pct90
9,15-1221,Computer and Information Research Scientists,2021,131490,142650,NaN


#### Normalisation

OEWS's occupation codes and titles are already standardised by BLS; the one classification change that occurs across years — the 2018 SOC revision — is handled as a coverage gap under Missing values above, rather than as a renaming or mapping step.

#### Outliers and invalid values

Year-over-year change in employment and median wage is checked per occupation, not pooled, since normal moves in this data run single digits to low teens percent. Anything beyond ±20% is flagged `is_outlier_yoy` — a threshold meant to catch a change in survey definition more than a genuine trend. Data Scientists trips it three years running (2022-2024), which, given the code only entered the survey in 2021, reads less like a definitional artifact and more like a genuinely fast-growing occupation still stabilising after its first full year. Year-over-year comparison suits an annual panel like this because there is only one observation per occupation per year, so there is no within-year distribution to compare against — the occupation's own prior year is the only available baseline.

In [46]:
oews_trend.loc[oews_trend["is_outlier_yoy"], ["occ_code", "occ_title", "year", "emp_yoy_pct", "wage_yoy_pct"]]

,occ_code,occ_title,year,emp_yoy_pct,wage_yoy_pct
29,15-2051,Data Scientists,2022,50.622759,2.566644
30,15-2051,Data Scientists,2023,20.722922,4.367150
31,15-2051,Data Scientists,2024,21.135385,4.230698


#### Discretisation and scaling

OEWS's annual files carry one row per occupation-year across just six target occupations — far too few rows to bin into meaningful terciles or quartiles, so no discretisation or scaling is applied here.

#### Stage summary

Row counts for the OEWS annual pipeline are shown in the consolidated Pipeline Summary below, alongside the other four sources.

## Pipeline Summary

The table below shows, for every source, how many rows went in, how many came out, and what was dropped versus flagged along the way. Nothing is silently discarded — a row is either kept, or it is dropped for a specific, reported reason (an exact duplicate, or out of scope for the target occupations), or it is kept and flagged for review.

In [47]:
bls_raw_counts = {name: len(flatten(latest_file(pattern)))
                   for name, pattern in [("jolts", "jolts_*.json"), ("ces", "ces_*.json"), ("oews", "oews_*.json")]}
bls_frames = [bls_jolts, bls_ces, bls_oews]
bls_flagged = sum(int(df["is_outlier"].sum()) + int(df["value_suspect"].sum()) for df in bls_frames)

indeed_frames = [indeed_ai_share, indeed_national, indeed_sector, indeed_metro, indeed_state]
indeed_total_rows = sum(len(f) for f in indeed_frames)
indeed_total_flagged = sum(int(f[c].sum()) for f in indeed_frames for c in f.columns
                            if c.startswith("is_outlier") or c.endswith("_suspect"))

oews_raw_total = sum(len(load_year(f)) for f in sorted(OEWS_XLSX_DIR.glob("national_M*_dl.xlsx")))

pipeline_summary = pd.DataFrame([
    {"source": "Hacker News", "rows_in": len(records), "rows_out": len(clean_df),
     "dropped": len(dropped), "flagged": int(clean_df["is_junk"].sum()),
     "notes": "dropped = dead/deleted comments; flagged = is_junk"},
    {"source": "ATS Job Postings", "rows_in": len(raw_ats), "rows_out": len(ats_final),
     "dropped": len(raw_ats) - len(ats_final),
     "flagged": int(ats_final[["is_near_dup", "desc_too_short", "desc_too_long"]].fillna(False).any(axis=1).sum()
                     + (ats_final["salary_suspect"] == True).sum()),
     "notes": "dropped = exact jobId duplicates; flagged = near-dup/salary/desc-length flags"},
    {"source": "Indeed Hiring Lab", "rows_in": indeed_total_rows, "rows_out": indeed_total_rows,
     "dropped": 0, "flagged": indeed_total_flagged,
     "notes": "5 files, reshaping only; flagged = outlier + range-check rows across all files"},
    {"source": "BLS Time Series", "rows_in": sum(bls_raw_counts.values()),
     "rows_out": len(bls_jolts) + len(bls_ces) + len(bls_oews),
     "dropped": sum(bls_raw_counts.values()) - (len(bls_jolts) + len(bls_ces) + len(bls_oews)),
     "flagged": bls_flagged,
     "notes": "dropped = M13 annual-average rows; flagged = outlier + negative-value rows"},
    {"source": "OEWS Annual Files", "rows_in": oews_raw_total, "rows_out": len(oews_trend),
     "dropped": oews_raw_total - len(oews_trend),
     "flagged": int(oews_trend["wage_topcoded"].sum() + oews_trend["is_outlier_yoy"].sum()),
     "notes": "dropped = filtered down to the 6 target SOC codes; flagged = topcoded wage + YoY outliers"},
])
pipeline_summary

,source,rows_in,rows_out,dropped,flagged,notes
0,Hacker News,52040,48296,3744,233,dropped = dead/deleted comments; flagged = is_junk
1,ATS Job Postings,4409,4294,115,1025,dropped = exact jobId duplicates; flagged = near-dup/salary/desc-length flags
2,Indeed Hiring Lab,1801661,1801661,0,9022,"5 files, reshaping only; flagged = outlier + range-check rows across all files"
3,BLS Time Series,1956,1956,0,12,dropped = M13 annual-average rows; flagged = outlier + negative-value rows
4,OEWS Annual Files,9670,33,9637,4,dropped = filtered down to the 6 target SOC codes; flagged = topcoded wage + YoY outliers


## Exploratory Visualizations